# 〰️ Beyond Linearity
**ISLP Chapter 7 · Pattern Recognition for the Rest of Us**

> Linear regression assumes a straight-line relationship. But many real-world relationships are curved. This notebook covers flexible extensions: polynomial regression, splines, and Generalized Additive Models — all interpretable ways to model nonlinearity.

### What you'll learn
- Polynomial regression: fitting curves with OLS
- Step functions: piecewise constants
- Regression splines: smooth curves with knots
- Natural cubic splines
- GAMs: additive models with one smooth term per feature

### Dataset: Wage (ISLP) + Auto
### Time: ~55 minutes

In [ ]:
# Overview: income vs age relationship
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(wage['age'], wage['wage'], alpha=0.15, color='#888', s=10)
ax.set_xlabel('Age'); ax.set_ylabel('Wage ($)')
ax.set_title('Wage vs Age — clearly nonlinear, needs more than a straight line')
plt.tight_layout(); plt.show()

## 📐 Part 1 — Polynomial Regression

Extend linear regression with polynomial terms:
```
Y = β₀ + β₁X + β₂X² + β₃X³ + ... + βdXd + ε
```
This is still linear regression (linear in the β coefficients) — OLS works perfectly. The trick is just creating the polynomial features from X.

In [ ]:
# Polynomial regression: compare degrees
X_age = wage['age'].values.reshape(-1,1)
y_wage = wage['wage'].values
age_range = np.linspace(wage['age'].min(), wage['age'].max(), 200).reshape(-1,1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
degrees = [1, 2, 4, 10]
for ax, d in zip(axes, degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#1e5fa8', lw=2.5)
    ax.set_title(f'Degree {d}\n10-fold CV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Polynomial Regression: Increasing Degree', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Degree 4 captures the curve. Degree 10 wiggles — overfitting at the extremes')

## 🪢 Part 2 — Splines: Smooth Piecewise Polynomials

Polynomials are global — a wiggle at one end affects predictions everywhere. **Splines** are piecewise polynomials that join smoothly at **knots**.

A **regression spline** with K knots has K+d+1 basis functions (d = degree). The result: flexibility where the data is dense, smoothness everywhere.

**Natural cubic splines** add the constraint that the function is linear beyond the boundary knots — preventing wild behavior at the extremes.

In [ ]:
# Compare: linear, polynomial, spline
from sklearn.preprocessing import SplineTransformer

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_spline = [
    ('Degree-4 Poly', Pipeline([('poly',PolynomialFeatures(4)),('lr',LinearRegression())])),
    ('Spline (4 knots)', Pipeline([('spline',SplineTransformer(n_knots=4, degree=3)),('lr',LinearRegression())])),
    ('Spline (8 knots)', Pipeline([('spline',SplineTransformer(n_knots=8, degree=3)),('lr',LinearRegression())])),
]

for ax, (title, pipe) in zip(axes, models_spline):
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#e85d20', lw=2.5)
    ax.set_title(f'{title}\nCV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Splines vs Polynomial', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## ➕ Part 3 — Generalized Additive Models (GAMs)

GAMs extend linear models to allow nonlinear relationships:
```
Y = β₀ + f₁(X₁) + f₂(X₂) + ... + fₚ(Xₚ) + ε
```
Each fⱼ is a smooth function estimated from the data. The **additive** structure means:
- Each feature contributes independently
- Interpretable: plot each fⱼ to see its effect
- Much more flexible than linear but more interpretable than neural networks

In [ ]:
!pip install pygam -q
from pygam import LinearGAM, s, f, l

# GAM: wage ~ s(age) + s(year) + education
wage_enc = pd.get_dummies(wage[['age','year','education','wage']], drop_first=True, dtype=float)
feature_cols = [c for c in wage_enc.columns if c != 'wage']
X_gam = wage_enc[feature_cols].values
y_gam = wage_enc['wage'].values

gam = LinearGAM(s(0) + s(1) + l(2) + l(3) + l(4) + l(5)).fit(X_gam, y_gam)
print(f'GAM R²: {gam.statistics_["pseudo_r2"]["McFadden"]:.4f}')
gam.summary()

In [ ]:
# Plot GAM smooth terms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, term_idx, label in zip(axes, [0, 1], ['age', 'year']):
    XX = gam.generate_X_grid(term=term_idx)
    pdep, confi = gam.partial_dependence(term=term_idx, width=0.95)
    ax.plot(XX[:,term_idx], pdep, color='#1e5fa8', lw=2.5)
    ax.fill_between(XX[:,term_idx], confi[:,0], confi[:,1], alpha=0.2, color='#1e5fa8')
    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel(label); ax.set_ylabel(f'Effect on wage')
    ax.set_title(f'GAM: smooth term for {label}')
plt.suptitle('GAM Partial Effects — Age and Year', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Wage rises steeply with age until ~40, then levels off')
print('   Year shows a gradual upward trend — wages rising over time')

In [ ]:
# @title 📝 Quiz — Beyond Linearity
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is a regression spline and how does it differ from a polynomial?
# @markdown **Q2:** What are knots in the context of splines?
# @markdown **Q3:** What is the additive assumption in GAMs?
# @markdown **Q4:** How do you choose the number/location of knots?
# @markdown **Q5:** Name one advantage of splines over high-degree polynomials

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Beyond Linearity"
# @title 🤖 AI Feedback — run this cell for instant grading
# @markdown **Enter your GitHub username below, then click ▶ Run**
# @markdown
# @markdown No API key needed — AI grading runs directly inside Colab for free.
# @markdown
# @markdown *(If AI grading is unavailable, you still get completion-based feedback)*

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ─────────────────────────────────────────────────────────────
import json, textwrap, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade_with_gemini(answers_dict, nb_name):
    """Call Gemini inside Colab — no API key required."""
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)

    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )

    prompt = f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

Student answers ({n_done}/{n_total} answered):
{qa_lines if qa_lines else "(none provided)"}

Grade conceptual understanding, not exact wording. Accept reasonable paraphrases.
Be encouraging and specific. Reply ONLY in this JSON format, no markdown:
{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions>",
  "tip": "<one specific thing to remember from this technique>"
}}"""

    # ── Attempt 1: Colab-native Gemini (zero config) ──────────
    try:
        import google.generativeai as genai
        import google.auth
        # Use Colab's ambient credentials — no API key needed
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        model  = genai.GenerativeModel("gemini-2.0-flash")
        result = model.generate_content(prompt)
        return result.text, "Gemini (Colab)"
    except Exception:
        pass

    # ── Attempt 2: Colab userdata key (if student added one) ──
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key and len(api_key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=api_key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini (API key)"
    except Exception:
        pass

    # ── Attempt 3: urllib fallback ─────────────────────────────
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key and len(api_key) > 10:
            import urllib.request
            url = (f"https://generativelanguage.googleapis.com/v1beta/"
                   f"models/gemini-2.0-flash:generateContent?key={api_key}")
            payload = json.dumps({
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
            }).encode()
            req = urllib.request.Request(url, data=payload,
                                          headers={"Content-Type":"application/json"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
                return data["candidates"][0]["content"]["parts"][0]["text"], "Gemini (urllib)"
    except Exception:
        pass

    return None, None

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "feedback":"No answers provided — fill in the quiz answers above and re-run.",
                "tip":"Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score":score,"grade":"Needs Review",
                "feedback":f"You answered {n_answered}/{n_total} questions but responses are very brief. Try to explain your reasoning in 1-2 sentences.",
                "tip":"Aim for a sentence or two per answer — quality over brevity."}
    elif n_answered == n_total:
        return {"quiz_score":score,"grade":"Good",
                "feedback":f"All {n_total} questions answered with good detail.",
                "tip":"Review any concepts that felt unclear and check the Further Reading links."}
    else:
        return {"quiz_score":score,"grade":"Needs Review",
                "feedback":f"{n_answered} of {n_total} questions answered. Complete the remaining {n_total-n_answered} for full credit.",
                "tip":"Answer all questions before submitting."}

def _print_result(result, username, nb_name, source):
    colors = {"Excellent":"\033[92m","Good":"\033[94m",
              "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R     = "\033[0m"
    grade = result.get("grade","See feedback")
    C     = colors.get(grade,"\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by  {source}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    for line in textwrap.wrap(result.get("feedback",""), width=56):
        print(f"  {line}")
    tip = result.get("tip","")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{len(answers)} provided")
    if username: print(f"  Student  : @{username}")

    raw, source = _grade_with_gemini(answers, _NB_TITLE)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            result = {"quiz_score":n_answered,
                      "grade":"Good" if n_answered==len(answers) else "Needs Review",
                      "feedback":raw[:400],"tip":""}
    else:
        print("  AI unavailable \u2014 using completion-based feedback\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE, source)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


In [ ]:
_NB_NAME_ = "Beyond Linearity"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
